# 🇮🇹 Everyday Italian — Pimsleur-style audio lesson generator (B1)

**Lesson 7:** weather & forecasts · holidays · climate change · health (doctor, pharmacy) · education · engineering & science at work · artificial intelligence · everyday technology · cinema · social media · fashion

Grammar in context: future for forecasts, *stare per* + inf., second-type conditional (*se potessi, partirei*), *ho paura che / sembra che / è possibile che* + subjunctive, gerund (*usando l'IA*), *sempre più*, *di cui*, *mi è piaciuto*, *mi fa male / mi fanno male*. Part 12 reviews sentences from Lessons 1–6.

How the audio works:
* **English cue → pause → Italian answer → pause.** The first pause is for you to *say* the sentence, the second for you to *repeat* after the native speaker.
* **Pause lengths depend on the Italian.** Each pause is computed from the measured duration of the Italian clip:
  * answer pause = `PAUSE_SCALE × clamp(THINK_MS + SAY_FACTOR × d_it)`
  * repeat pause = `PAUSE_SCALE × clamp(REPEAT_BASE_MS + REPEAT_FACTOR × d_it)`
* **Two Italian voices** (female/male, Microsoft neural voices via `edge-tts`) plus an English narrator; falls back to `gTTS` automatically if a clip fails.
* Back-chaining (`ingegneria → gegneria → ...`), repeated words (`x2`, `x3`, optional slow repeat), spaced review of earlier sentences, role-play, and the dialogue at start and end.

**Run:** Runtime → Run all. The MP3 and a time-stamped transcript are downloaded at the end (~3–5 min synthesis for ~420 clips; the cache makes reruns fast).


In [ ]:
!pip -q install edge-tts gTTS pydub
# ffmpeg is pre-installed on Colab (pydub needs it)

## 1 · Settings

In [ ]:
#@title Voices, speed and pause settings { display-mode: "form" }
LESSON_NAME   = "Lesson07_Mondo_B1"  #@param {type:"string"}
ENGINE        = "edge"   #@param ["edge", "gtts"]
VOICE_EN      = "en-US-AndrewNeural"  #@param ["en-US-AndrewNeural", "en-US-GuyNeural", "en-US-JennyNeural", "en-GB-RyanNeural"]
VOICE_IT_F    = "it-IT-ElsaNeural"    #@param ["it-IT-ElsaNeural", "it-IT-IsabellaNeural"]
VOICE_IT_M    = "it-IT-DiegoNeural"   #@param ["it-IT-DiegoNeural", "it-IT-GiuseppeNeural"]
EN_RATE       = "-5%"    #@param {type:"string"}
IT_RATE       = "-10%"   #@param {type:"string"}
SLOW_RATE     = "-35%"   #@param {type:"string"}

#@markdown **Pauses** (ms).  `d_it` = duration of the Italian clip.
PAUSE_SCALE    = 1.0    #@param {type:"slider", min:0.6, max:1.8, step:0.05}
THINK_MS       = 1500   #@param {type:"integer"}
SAY_FACTOR     = 1.5    #@param {type:"number"}
MIN_ANSWER_MS  = 2500   #@param {type:"integer"}
MAX_ANSWER_MS  = 12000  #@param {type:"integer"}
REPEAT_BASE_MS = 1000   #@param {type:"integer"}
REPEAT_FACTOR  = 1.3    #@param {type:"number"}
MIN_REPEAT_MS  = 1800   #@param {type:"integer"}
MAX_REPEAT_MS  = 10000  #@param {type:"integer"}
USE_PIANO      = True   #@param {type:"boolean"}

CFG = dict(ENGINE=ENGINE, VOICE_EN=VOICE_EN, VOICE_IT_F=VOICE_IT_F, VOICE_IT_M=VOICE_IT_M,
           EN_RATE=EN_RATE, IT_RATE=IT_RATE, SLOW_RATE=SLOW_RATE, PAUSE_SCALE=PAUSE_SCALE,
           THINK_MS=THINK_MS, SAY_FACTOR=SAY_FACTOR, MIN_ANSWER_MS=MIN_ANSWER_MS, MAX_ANSWER_MS=MAX_ANSWER_MS,
           REPEAT_BASE_MS=REPEAT_BASE_MS, REPEAT_FACTOR=REPEAT_FACTOR,
           MIN_REPEAT_MS=MIN_REPEAT_MS, MAX_REPEAT_MS=MAX_REPEAT_MS,
           NARRATION_GAP_MS=700, DIALOGUE_GAP_MS=600, SECTION_GAP_MS=1200,
           # dry-run speech-rate model (calibrated on the edge-tts voices above)
           IT_SYL_PER_S=4.6, EN_WORDS_PER_S=2.7, CLIP_PAD_MS=570,
           CACHE_DIR="/content/tts_cache", USE_PIANO=USE_PIANO,
           PIANO_URL="https://csee.umbc.edu/~simsek/shortpiano.mp3")

## 2 · TTS engine

In [ ]:
import os, re, json, time, hashlib, subprocess, tempfile, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pydub import AudioSegment

# ----------------------------------------------------------------------------
# Script tags (one per line; blank lines and lines starting with # are ignored)
#   (section) Title                 -> marker in the transcript only (no audio)
#   (narrator) English text         -> English narrator. {curly braces} = Italian
#                                      inserted inside the sentence, e.g.
#                                      "The word {ricevimento} means office hours."
#   (it F) testo  / (it M) testo    -> Italian line (dialogue), female / male voice
#   (repeat [x2] [slow] [F|M]) testo-> Italian model + pause for you to REPEAT.
#                                      x2/x3 = say it 2-3 times; slow = the extra
#                                      repetitions are spoken slowly.
#   (prompt [x2] [F|M]) English || Italiano
#                                   -> English cue, pause to ANSWER (length set by
#                                      the Italian), Italian answer, pause to REPEAT.
#   (build) zione | iscrizione      -> back-chaining: each piece + repeat pause
#   (pause 3)                       -> silence in seconds
#   (english) / (italian)           -> old tags still work (= narrator / it F)
# ----------------------------------------------------------------------------

VOWELS = "aeiouàèéìíòóùúy"

def italian_syllables(text):
    """Rough Italian syllable count (vowel groups); used for dry-run timing only."""
    words = re.findall(r"[A-Za-zÀ-ÿ']+", text.lower())
    n = 0
    for w in words:
        n += max(1, len(re.findall(f"[{VOWELS}]+", w)))
    return n


class PimsleurTTS:
    def __init__(self, cfg):
        self.cfg = cfg
        os.makedirs(cfg["CACHE_DIR"], exist_ok=True)
        self.piano_clip_path = self._download_piano_clip() if cfg.get("USE_PIANO") else None

    # ------------------------------------------------------------------ parsing
    def parse(self, script):
        events = []          # list of (kind, payload dict)
        voice_toggle = 0
        for raw in script.split("\n"):
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            m = re.match(r"^\(([^)]*)\)\s*(.*)$", line)
            if not m:
                print(f"⚠️ untagged line skipped: {line[:60]}")
                continue
            tag, body = m.group(1).strip().lower(), m.group(2).strip()
            words = tag.split()
            head = words[0]
            reps = 1
            for w in words:
                if re.fullmatch(r"x\d", w):
                    reps = int(w[1])
            slow = "slow" in words
            gender = "M" if "m" in words else ("F" if "f" in words else None)

            if head in ("section",):
                events.append(("section", {"title": body}))
            elif head in ("narrator", "english", "en"):
                events.append(("narrator", {"text": body}))
            elif head in ("it", "italian", "italiano"):
                events.append(("it", {"text": body, "g": gender or "F"}))
            elif head == "repeat":
                if gender is None:
                    gender = "FM"[voice_toggle % 2]; voice_toggle += 1
                events.append(("repeat", {"text": body, "g": gender, "reps": reps, "slow": slow}))
            elif head == "prompt":
                if "||" not in body:
                    print(f"⚠️ prompt without '||' skipped: {body[:60]}"); continue
                en, it = [s.strip() for s in body.split("||", 1)]
                if gender is None:
                    gender = "FM"[voice_toggle % 2]; voice_toggle += 1
                events.append(("prompt", {"en": en, "it": it, "g": gender, "reps": reps, "slow": slow}))
            elif head == "build":
                pieces = [p.strip().lstrip("-") for p in body.split("|") if p.strip()]
                events.append(("build", {"pieces": pieces, "g": gender or "F"}))
            elif head == "pause":
                sec = float(words[1]) if len(words) > 1 else float(body or 1)
                events.append(("pause", {"ms": int(sec * 1000)}))
            else:
                print(f"⚠️ unknown tag ({tag}) skipped")
        return events

    # ------------------------------------------------------------------ pauses
    def answer_pause(self, it_ms):
        c = self.cfg
        p = c["THINK_MS"] + c["SAY_FACTOR"] * it_ms
        return int(c["PAUSE_SCALE"] * min(max(p, c["MIN_ANSWER_MS"]), c["MAX_ANSWER_MS"]))

    def repeat_pause(self, it_ms):
        c = self.cfg
        p = c["REPEAT_BASE_MS"] + c["REPEAT_FACTOR"] * it_ms
        return int(c["PAUSE_SCALE"] * min(max(p, c["MIN_REPEAT_MS"]), c["MAX_REPEAT_MS"]))

    # ------------------------------------------------------------------ TTS jobs
    def _voice(self, lang, g):
        c = self.cfg
        if lang == "en":
            return c["VOICE_EN"]
        return c["VOICE_IT_M"] if g == "M" else c["VOICE_IT_F"]

    @staticmethod
    def _split_narration(text):
        """'The word {casa} means house' -> [('en','The word'),('it','casa'),('en','means house')]"""
        parts = []
        for i, chunk in enumerate(re.split(r"\{(.*?)\}", text)):
            chunk = chunk.strip()
            if re.search(r"\w", chunk):          # skip punctuation-only pieces
                parts.append(("it" if i % 2 else "en", chunk))
        return parts

    def jobs_for(self, events):
        """All unique (text, lang, gender, slow) clips the lesson needs."""
        jobs = set()
        for kind, d in events:
            if kind == "narrator":
                for lang, chunk in self._split_narration(d["text"]):
                    jobs.add((chunk, lang, "F", False))
            elif kind == "it":
                jobs.add((d["text"], "it", d["g"], False))
            elif kind in ("repeat", "prompt"):
                it = d["text"] if kind == "repeat" else d["it"]
                jobs.add((it, "it", d["g"], False))
                if d["slow"]:
                    jobs.add((it, "it", d["g"], True))
                if kind == "prompt":
                    jobs.add((d["en"], "en", "F", False))
            elif kind == "build":
                for p in d["pieces"]:
                    jobs.add((p, "it", d["g"], False))
        return sorted(jobs)

    def _cache_path(self, job):
        text, lang, g, slow = job
        key = json.dumps([text, self._voice(lang, g), slow, self.cfg["ENGINE"],
                          self.cfg["EN_RATE"] if lang == "en" else self.cfg["IT_RATE"],
                          self.cfg["SLOW_RATE"]])
        return os.path.join(self.cfg["CACHE_DIR"], hashlib.md5(key.encode()).hexdigest() + ".mp3")

    def _synth_edge(self, job, path):
        text, lang, g, slow = job
        rate = self.cfg["SLOW_RATE"] if slow else (self.cfg["EN_RATE"] if lang == "en" else self.cfg["IT_RATE"])
        cmd = ["edge-tts", f"--voice={self._voice(lang, g)}", f"--rate={rate}",
               f"--text={text}", f"--write-media={path}"]
        proxy = os.environ.get("HTTPS_PROXY") or os.environ.get("https_proxy")
        if proxy:                       # only matters behind a proxy; Colab has none
            cmd.insert(1, f"--proxy={proxy}")
        subprocess.run(cmd, check=True, capture_output=True, timeout=60)

    def _synth_gtts(self, job, path):
        from gtts import gTTS
        text, lang, g, slow = job
        gTTS(text=text, lang=lang, slow=slow).save(path)

    @staticmethod
    def _clip_ok(job, path):
        """Reject empty / truncated clips (the online TTS sometimes returns silence)."""
        if not (os.path.exists(path) and os.path.getsize(path) > 0):
            return False
        try:
            a = AudioSegment.from_file(path)
        except Exception:
            return False
        n = italian_syllables(job[0]) if job[1] == "it" else 1.5 * len(job[0].split())
        return len(a) >= 1000 * n / 12 and a.dBFS > -60     # far faster than any voice = broken

    def _synth_one(self, job):
        path = self._cache_path(job)
        if self._clip_ok(job, path):
            return job, path, None
        err = None
        engines = [self.cfg["ENGINE"]] + (["gtts"] if self.cfg["ENGINE"] != "gtts" else [])
        for eng in engines:
            for attempt in range(4):
                try:
                    (self._synth_edge if eng == "edge" else self._synth_gtts)(job, path)
                    if self._clip_ok(job, path):
                        return job, path, None
                    err = f"{eng}: empty or truncated audio"
                except Exception as e:
                    err = f"{eng}: {e}"
                time.sleep(2 * (attempt + 1))      # back off: the free TTS service rate-limits bursts
        return job, None, err

    def synthesize_all(self, events, workers=6):
        jobs = self.jobs_for(events)
        print(f"🗣️  {len(jobs)} unique clips to synthesize (cached ones are reused)...")
        self.clips, self.failed = {}, []
        with ThreadPoolExecutor(workers) as ex:
            for i, (job, path, err) in enumerate(ex.map(self._synth_one, jobs), 1):
                if path:
                    self.clips[job] = AudioSegment.from_file(path)
                else:
                    self.failed.append((job, err))
                    self.clips[job] = AudioSegment.silent(duration=1000)
                if i % 100 == 0 or i == len(jobs):
                    print(f"   {i}/{len(jobs)} done")
        if self.failed:
            print(f"⚠️ {len(self.failed)} clips failed (replaced by 1 s silence), e.g.: {self.failed[0]}")

    def clip(self, text, lang, g="F", slow=False):
        return self.clips[(text, lang, g, slow)]

    # ------------------------------------------------------------------ assembly
    def assemble(self, events):
        c = self.cfg
        S = AudioSegment.silent
        out = S(duration=300)
        transcript = []

        def stamp(label, text):
            ms = len(out)
            transcript.append(f"[{ms//60000:02d}:{(ms//1000)%60:02d}] {label:9s} {text}")

        for kind, d in events:
            if kind == "section":
                stamp("SECTION", "=== " + d["title"] + " ===")
                out += S(duration=c["SECTION_GAP_MS"])
            elif kind == "pause":
                out += S(duration=int(d["ms"] * c["PAUSE_SCALE"]))
            elif kind == "narrator":
                stamp("NARRATOR", d["text"])
                for lang, chunk in self._split_narration(d["text"]):
                    out += self.clip(chunk, lang) + S(duration=120)
                out += S(duration=c["NARRATION_GAP_MS"])
            elif kind == "it":
                stamp(f"IT-{d['g']}", d["text"])
                out += self.clip(d["text"], "it", d["g"]) + S(duration=c["DIALOGUE_GAP_MS"])
            elif kind == "repeat":
                stamp("REPEAT", d["text"] + (f"  (x{d['reps']})" if d["reps"] > 1 else ""))
                normal = self.clip(d["text"], "it", d["g"])
                for r in range(d["reps"]):
                    seg = self.clip(d["text"], "it", d["g"], True) if (d["slow"] and r > 0) else normal
                    out += seg + S(duration=self.repeat_pause(len(seg)))
            elif kind == "prompt":
                stamp("PROMPT", d["en"] + "  ->  " + d["it"])
                normal = self.clip(d["it"], "it", d["g"])
                out += self.clip(d["en"], "en") + S(duration=self.answer_pause(len(normal)))
                for r in range(d["reps"]):
                    seg = self.clip(d["it"], "it", d["g"], True) if (d["slow"] and r > 0) else normal
                    out += seg + S(duration=self.repeat_pause(len(seg)))
            elif kind == "build":
                stamp("BUILD", " | ".join(d["pieces"]))
                for p in d["pieces"]:
                    seg = self.clip(p, "it", d["g"])
                    out += seg + S(duration=self.repeat_pause(len(seg)))
        return out, transcript

    # ------------------------------------------------------------------ dry run
    def estimate(self, events):
        """Duration estimate without any TTS (speech-rate model)."""
        c = self.cfg
        pad = c["CLIP_PAD_MS"]          # TTS clips carry some silence at both ends
        it_ms = lambda t, slow=False: pad + 1000 * italian_syllables(t) / (c["IT_SYL_PER_S"] * (0.75 if slow else 1))
        en_ms = lambda t: pad + 1000 * len(t.split()) / c["EN_WORDS_PER_S"]
        total, counts = 0.0, {}
        for kind, d in events:
            counts[kind] = counts.get(kind, 0) + 1
            if kind == "section":
                total += c["SECTION_GAP_MS"]
            elif kind == "pause":
                total += d["ms"] * c["PAUSE_SCALE"]
            elif kind == "narrator":
                for lang, chunk in self._split_narration(d["text"]):
                    total += (en_ms(chunk) if lang == "en" else it_ms(chunk)) + 120
                total += c["NARRATION_GAP_MS"]
            elif kind == "it":
                total += it_ms(d["text"]) + c["DIALOGUE_GAP_MS"]
            elif kind in ("repeat", "prompt"):
                it = d["text"] if kind == "repeat" else d["it"]
                if kind == "prompt":
                    total += en_ms(d["en"]) + self.answer_pause(it_ms(it))
                for r in range(d["reps"]):
                    m = it_ms(it, d["slow"] and r > 0)
                    total += m + self.repeat_pause(m)
            elif kind == "build":
                for p in d["pieces"]:
                    m = it_ms(p); total += m + self.repeat_pause(m)
        return total / 1000, counts

    # ------------------------------------------------------------------ main
    def _download_piano_clip(self):
        url = self.cfg.get("PIANO_URL")
        try:
            print("🎹 Downloading piano intro/outro clip...")
            p = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False).name
            urllib.request.urlretrieve(url, p)
            return p
        except Exception as e:
            print(f"⚠️ Could not download piano clip: {e}")
            return None

    def build(self, script, output_file):
        events = self.parse(script)
        self.synthesize_all(events)
        print("🎧 Assembling lesson...")
        audio, transcript = self.assemble(events)
        if self.piano_clip_path and os.path.exists(self.piano_clip_path):
            piano = AudioSegment.from_file(self.piano_clip_path)[:10000]   # first 10 s
            intro = piano.fade_in(1500).fade_out(1500)
            outro = piano.fade_in(1000).fade_out(3000)
            audio = intro + AudioSegment.silent(duration=500) + audio + AudioSegment.silent(duration=800) + outro
        audio.export(output_file, format="mp3", bitrate="96k")
        txt = os.path.splitext(output_file)[0] + "_transcript.txt"
        with open(txt, "w", encoding="utf-8") as f:
            f.write("\n".join(transcript))
        print(f"✅ {output_file}: {len(audio)/60000:.1f} min   |   transcript: {txt}")
        return output_file, txt


## 3 · Lesson script

One line = one audio event. Tags:

| tag | what you hear |
|---|---|
| `(narrator) text {italiano}` | English narrator; text in `{ }` is read by the Italian voice |
| `(it F) …` / `(it M) …` | Italian line (dialogue), female / male |
| `(repeat [x2] [slow] [F/M]) …` | Italian model + repeat pause, 1–3 times; `slow` = extra repetitions spoken slowly |
| `(prompt [x2] [F/M]) English \|\| Italiano` | English cue → answer pause → Italian → repeat pause |
| `(build) zione \| iscrizione` | back-chaining, each piece + repeat pause |
| `(pause 2)` · `(section) Title` | silence · transcript marker |

Voices for `repeat`/`prompt` alternate F/M automatically unless `F` or `M` is given.

In [ ]:
LESSON = r"""
#####################################################################
# EVERYDAY ITALIAN — Lesson 7 (B1)
# Theme: weather, holidays, education, climate, health, engineering,
#        science, artificial intelligence, technology, cinema,
#        social media, fashion
# Grammar: future for forecasts, stare per + inf, da + time, second-type
#          conditional (se potessi, partirei), ho paura che / sembra che /
#          è possibile che + congiuntivo, gerund (cambiando argomento),
#          sempre più, relative "di cui", mi è piaciuto
#####################################################################

(section) Opening
(narrator) Everyday Italian. Lesson seven.
(narrator) Today you will talk about the world around you. The weather and the climate. Your health. Science, technology, and artificial intelligence. And finally cinema, social media, and fashion.
(narrator) As always, answer out loud in the pause. Then repeat after the Italian speaker.
(narrator) Listen to this conversation. Luca and his colleague Sara meet at the coffee machine at work.
(pause 1)
(it F) Che tempo da lupi! Piove da tre giorni.
(it M) Eh sì. E le previsioni dicono che domani nevicherà in montagna.
(it F) Non vedo l'ora di andare in vacanza. Se potessi, partirei domani!
(it M) Dove andrai quest'estate?
(it F) In Sardegna. Ma con questo clima pazzo, ho paura che faccia troppo caldo.
(it M) È vero, le estati sono sempre più calde. Il cambiamento climatico è un problema serio.
(it F) Cambiando argomento, hai provato il nuovo programma di intelligenza artificiale?
(it M) Sì, lo uso per scrivere il codice. È utilissimo, ma a volte sbaglia.
(it F) Secondo me, non sostituirà mai gli ingegneri.
(it M) Speriamo! Senti, stasera esce il nuovo film di Sorrentino. Ti va di andarci?
(it F) Volentieri! Ma prima passo dal medico. Ho un mal di gola terribile.
(it M) Allora riposati. Ci vediamo alle nove davanti al cinema.
(pause 1.5)

(section) Part 1 — Weather
(narrator) Let's look at this conversation, piece by piece.
(narrator) {Che tempo da lupi} means: what terrible weather! Literally, weather for wolves.
(repeat x2 slow) Che tempo da lupi!
(prompt) Say: what terrible weather! Use the wolves. || Che tempo da lupi!
(prompt) Say: it has been raining for three days. || Piove da tre giorni.
(narrator) The weather forecast is {le previsioni del tempo}. Use the future for forecasts.
(build) sioni | previsioni | le previsioni del tempo
(prompt) Say: the forecast says it will snow in the mountains tomorrow. || Le previsioni dicono che domani nevicherà in montagna.
(prompt) Ask: what's the weather like today? || Che tempo fa oggi?
(prompt) Say: it's cold and there is fog. || Fa freddo e c'è nebbia.
(prompt) Say: tomorrow there will be thunderstorms in the north. Use {temporali}. || Domani ci saranno temporali al nord.
(prompt) Say: in the south it will be sunny. Use {ci sarà il sole}. || Al sud ci sarà il sole.
(prompt) Say: the temperature will reach thirty-five degrees. || La temperatura arriverà a trentacinque gradi.
(narrator) {Stare per} means: to be about to.
(prompt) Say: it's about to rain. Take an umbrella! Informal. || Sta per piovere. Prendi l'ombrello!
(prompt) Say: what terrible weather! || Che tempo da lupi!

(section) Part 2 — Holidays
(narrator) {Non vedo l'ora} means: I can't wait.
(repeat x2) Non vedo l'ora!
(prompt) Say: I can't wait to go on holiday. || Non vedo l'ora di andare in vacanza.
(narrator) Now a new structure. For imaginary situations, Italian uses {se} with the imperfect subjunctive, and then the conditional. If I could, I would leave.
(repeat x2 slow) Se potessi, partirei domani!
(prompt) Say: if I could, I would leave tomorrow! || Se potessi, partirei domani!
(prompt) Say: if I had more time, I would travel more. || Se avessi più tempo, viaggerei di più.
(prompt) Say: if I were rich, I would buy a house in Tuscany. || Se fossi ricco, comprerei una casa in Toscana.
(prompt) Ask a friend: where will you go this summer? || Dove andrai quest'estate?
(prompt) Answer: to Sardinia, for two weeks. || In Sardegna, per due settimane.
(prompt) Say: I prefer the mountains to the sea. || Preferisco la montagna al mare.
(prompt) Say: last year we went camping. || L'anno scorso siamo andati in campeggio.
(prompt) Say: the forecast says it will snow in the mountains tomorrow. || Le previsioni dicono che domani nevicherà in montagna.

(section) Part 3 — Climate
(narrator) Climate change is {il cambiamento climatico}. Global warming is {il riscaldamento globale}.
(build) matico | climatico | cambiamento climatico
(prompt) Say: climate change is a serious problem. || Il cambiamento climatico è un problema serio.
(narrator) {Sempre più} means: more and more.
(prompt) Say: summers are hotter and hotter. || Le estati sono sempre più calde.
(narrator) After {ho paura che}, I'm afraid that, use the subjunctive.
(prompt) Say: I'm afraid it will be too hot. || Ho paura che faccia troppo caldo.
(prompt) Say: this year there was a drought. Use {la siccità}. || Quest'anno c'è stata la siccità.
(prompt) Say: in autumn there were floods. Use {le alluvioni}. || In autunno ci sono state le alluvioni.
(narrator) Renewable energy is {le energie rinnovabili}.
(prompt) Say: we must reduce emissions. || Dobbiamo ridurre le emissioni.
(prompt) Say: we installed solar panels on the roof. || Abbiamo installato i pannelli solari sul tetto.
(prompt) Say: if I had more time, I would travel more. || Se avessi più tempo, viaggerei di più.
(prompt) Say: I can't wait to go on holiday. || Non vedo l'ora di andare in vacanza.

(section) Part 4 — Health
(narrator) Now health. The family doctor is {il medico di base}. To stop by is {passare da}.
(prompt) Say: first I'm stopping by the doctor's. || Prima passo dal medico.
(prompt) Say: I have a terrible sore throat. || Ho un mal di gola terribile.
(narrator) To say what hurts, use {mi fa male} for one thing, and {mi fanno male} for many.
(prompt) Say: my head hurts. || Mi fa male la testa.
(prompt) Say: my legs hurt. || Mi fanno male le gambe.
(prompt) Say: I've had a fever since yesterday. || Ho la febbre da ieri.
(prompt) The doctor asks, formally: do you have a cough? Use {la tosse}. || Ha la tosse?
(prompt) The doctor says, formally: rest and drink a lot of water. || Si riposi e beva molta acqua.
(narrator) The prescription is {la ricetta}. You take it to the pharmacy, {la farmacia}.
(prompt) Say: the doctor gave me a prescription for an antibiotic. || Il medico mi ha dato la ricetta per un antibiotico.
(prompt) Ask the pharmacist: do you have something for a cold? Use {il raffreddore}. || Ha qualcosa per il raffreddore?
(prompt) Say: in an emergency, go to the emergency room. Use {il pronto soccorso}. || In caso di emergenza, vai al pronto soccorso.
(prompt) Say to a friend: get well soon! || Guarisci presto!
(prompt) Say: I'm afraid it will be too hot. || Ho paura che faccia troppo caldo.
(prompt) Say: summers are hotter and hotter. || Le estati sono sempre più calde.

(section) Part 5 — Education
(narrator) Education is {l'istruzione}. In Italy, children go to primary school, {la scuola elementare}, then middle school, {la scuola media}, then high school, {le superiori}.
(prompt) Say: in Italy, school is compulsory until sixteen. || In Italia la scuola è obbligatoria fino a sedici anni.
(prompt) Say: my son goes to middle school. || Mio figlio va alla scuola media.
(prompt) Say: in my opinion, public education is very important. || Secondo me, l'istruzione pubblica è molto importante.
(prompt) Say: teachers should earn more. || Gli insegnanti dovrebbero guadagnare di più.
(prompt) Say: I'm taking an online course on data analysis. || Sto seguendo un corso online di analisi dei dati.
(prompt) Say: one never stops learning. || Non si smette mai di imparare.
(prompt) Say: I've had a fever since yesterday. || Ho la febbre da ieri.
(prompt) Ask the pharmacist: do you have something for a cold? || Ha qualcosa per il raffreddore?

(section) Part 6 — Engineering and science
(narrator) Now your work. To design is {progettare}. A prototype is {un prototipo}. A deadline is {una scadenza}.
(prompt) Say: I work as an engineer in the photonics sector. || Lavoro come ingegnere nel settore della fotonica.
(prompt) Say: we are designing a new optical chip. || Stiamo progettando un nuovo chip ottico.
(prompt) Say: we tested the prototype in the lab. || Abbiamo testato il prototipo in laboratorio.
(prompt) Say: the results are better than expected. || I risultati sono migliori del previsto.
(prompt) Say: the deadline for the project is in March. || La scadenza del progetto è a marzo.
(narrator) Research is {la ricerca}. A discovery is {una scoperta}. A scientific article is {un articolo scientifico}.
(build) perta | scoperta | una scoperta
(prompt) Say: I do research on lasers. || Faccio ricerca sui laser.
(prompt) Say: we published an article in an international journal. || Abbiamo pubblicato un articolo su una rivista internazionale.
(prompt) Say: it's an important discovery. || È una scoperta importante.
(narrator) {Sembra che} means: it seems that. It takes the subjunctive.
(prompt) Say: it seems the experiment works. || Sembra che l'esperimento funzioni.
(prompt) Say: one never stops learning. || Non si smette mai di imparare.
(prompt) Say: if I could, I would leave tomorrow! || Se potessi, partirei domani!

(section) Part 7 — Artificial intelligence
(narrator) Artificial intelligence is {l'intelligenza artificiale}. Many Italians just say {l'IA}.
(build) ciale | artificiale | intelligenza artificiale
(prompt) Ask a colleague: have you tried the new AI program? || Hai provato il nuovo programma di intelligenza artificiale?
(prompt) Say: I use it to write code. || Lo uso per scrivere il codice.
(prompt) Say: it's very useful, but sometimes it makes mistakes. || È utilissimo, ma a volte sbaglia.
(prompt) Say: in my opinion, it will never replace engineers. || Secondo me, non sostituirà mai gli ingegneri.
(prompt) Say: some people are afraid that AI will replace many jobs. || Alcuni hanno paura che l'IA sostituisca molti lavori.
(prompt) Say: it's possible that it will change the way we work. || È possibile che cambi il nostro modo di lavorare.
(narrator) The gerund, ending in {ando} or {endo}, means "by doing" something.
(prompt) Say: by using AI, I save a lot of time. || Usando l'IA, risparmio molto tempo.
(prompt) Say: we always have to check the results. || Dobbiamo sempre controllare i risultati.
(prompt) Say: we are designing a new optical chip. || Stiamo progettando un nuovo chip ottico.
(prompt) Say: my head hurts. || Mi fa male la testa.

(section) Part 8 — Technology
(narrator) Now everyday technology. To download is {scaricare}. To update is {aggiornare}. The charger is {il caricabatterie}.
(prompt) Say: I downloaded a new app. || Ho scaricato una nuova app.
(prompt) Say: I have to update my phone. Use {il telefono}. || Devo aggiornare il telefono.
(prompt) Say: my computer froze. Use {bloccarsi}. || Il computer si è bloccato.
(prompt) Say: I forgot my password. || Ho dimenticato la password.
(prompt) Ask: do you have a phone charger? || Hai un caricabatterie per il telefono?
(prompt) Say: my battery is almost dead. Use {scarica}. || La batteria è quasi scarica.
(prompt) Ask: have you tried turning it off and on again? || Hai provato a spegnerlo e riaccenderlo?
(prompt) Say: by using AI, I save a lot of time. || Usando l'IA, risparmio molto tempo.
(prompt) Say: it's possible that it will change the way we work. || È possibile che cambi il nostro modo di lavorare.

(section) Part 9 — Cinema
(narrator) Now the cinema. A film director is {un regista}. The subtitles are {i sottotitoli}.
(prompt) Say: tonight the new Sorrentino film is coming out. || Stasera esce il nuovo film di Sorrentino.
(narrator) {Ti va di} means: do you feel like?
(prompt) Ask a friend: do you feel like going? Use {andarci}. || Ti va di andarci?
(prompt) Answer: I'd love to! || Volentieri!
(prompt) Say: see you at nine in front of the cinema. || Ci vediamo alle nove davanti al cinema.
(prompt) Say: I prefer to watch films in the original language with subtitles. || Preferisco vedere i film in lingua originale con i sottotitoli.
(narrator) To say you liked something, use {mi è piaciuto}, or {mi è piaciuta} for a feminine noun.
(prompt) Say: I liked the film a lot. || Il film mi è piaciuto molto.
(prompt) Say: the ending didn't convince me. Use {il finale} and {convincere}. || Il finale non mi ha convinto.
(prompt) Say: Fellini is my favourite director. || Fellini è il mio regista preferito.
(prompt) Say: this is the film I told you about. Use {di cui}. || Questo è il film di cui ti ho parlato.
(prompt) Say: my computer froze. || Il computer si è bloccato.

(section) Part 10 — Social media
(narrator) Social media in Italian is {i social}. To post is {pubblicare}. A like is {un mi piace}.
(prompt) Say: I posted a photo of the holiday. || Ho pubblicato una foto della vacanza.
(prompt) Say: my post got a hundred likes. || Il mio post ha ricevuto cento mi piace.
(prompt) Say: I spend too much time on social media. || Passo troppo tempo sui social.
(prompt) Say: there is a lot of fake news online. Use {notizie false}. || Online ci sono molte notizie false.
(prompt) Say: you have to check the sources. || Bisogna verificare le fonti.
(prompt) Say: I deleted the app from my phone. || Ho cancellato l'app dal telefono.
(prompt) Say: this is the film I told you about. || Questo è il film di cui ti ho parlato.
(prompt) Say: I downloaded a new app. || Ho scaricato una nuova app.

(section) Part 11 — Fashion
(narrator) Finally, fashion: {la moda}. Milan is the Italian capital of fashion. Every year there is {la settimana della moda}, fashion week.
(prompt) Say: Milan is the capital of fashion. || Milano è la capitale della moda.
(prompt) Say to a friend: that jacket looks great on you! || Quella giacca ti sta benissimo!
(prompt) Say: this colour is in fashion this year. || Questo colore va di moda quest'anno.
(prompt) Say: these shoes are out of fashion. || Queste scarpe sono fuori moda.
(prompt) Say: I prefer to dress casually. Use {vestirsi}. || Preferisco vestirmi in modo casual.
(prompt) Say: more and more people buy second-hand clothes. || Sempre più persone comprano vestiti di seconda mano.
(prompt) Say: sustainable fashion is important for the planet. || La moda sostenibile è importante per il pianeta.
(prompt) Say: I spend too much time on social media. || Passo troppo tempo sui social.
(prompt) Say: climate change is a serious problem. || Il cambiamento climatico è un problema serio.

(section) Part 12 — Review
(narrator) Let's review. First, sentences from earlier lessons.
(prompt) Say: I would like to do my thesis on photonics. || Vorrei fare la tesi sulla fotonica.
(prompt) Say: I would like two hundred grams of it. || Ne vorrei due etti.
(prompt) Say: the flight was cancelled. || Il volo è stato cancellato.
(prompt) Say: I thought it was still yellow. || Pensavo che fosse ancora giallo.
(prompt) Say: I am really happy that you are happy. || Sono proprio contento che tu sia felice.
(prompt) Say: tomorrow I'm getting my hair cut. || Domani mi faccio tagliare i capelli.
(narrator) Now today's lesson. Answer quickly.
(prompt) Say: it has been raining for three days. || Piove da tre giorni.
(prompt) Say: it's about to rain. || Sta per piovere.
(prompt) Say: if I had more time, I would travel more. || Se avessi più tempo, viaggerei di più.
(prompt) Say: we must reduce emissions. || Dobbiamo ridurre le emissioni.
(prompt) Say: I have a terrible sore throat. || Ho un mal di gola terribile.
(prompt) Say: teachers should earn more. || Gli insegnanti dovrebbero guadagnare di più.
(prompt) Say: we tested the prototype in the lab. || Abbiamo testato il prototipo in laboratorio.
(prompt) Say: some people are afraid that AI will replace many jobs. || Alcuni hanno paura che l'IA sostituisca molti lavori.
(prompt) Say: I forgot my password. || Ho dimenticato la password.
(prompt) Say: I liked the film a lot. || Il film mi è piaciuto molto.
(prompt) Say: you have to check the sources. || Bisogna verificare le fonti.
(prompt) Say: that jacket looks great on you! || Quella giacca ti sta benissimo!

(narrator) Now you are Luca. Answer Sara.
(it F) Che tempo da lupi! Piove da tre giorni.
(prompt M) Say: yes. And the forecast says it will snow in the mountains tomorrow. || Eh sì. E le previsioni dicono che domani nevicherà in montagna.
(it F) Hai provato il nuovo programma di intelligenza artificiale?
(prompt M) Say: yes, I use it to write code. It's very useful, but sometimes it makes mistakes. || Sì, lo uso per scrivere il codice. È utilissimo, ma a volte sbaglia.
(it F) Secondo me, non sostituirà mai gli ingegneri.
(prompt M) Say: let's hope so! Tonight the new Sorrentino film is coming out. Do you feel like going? || Speriamo! Stasera esce il nuovo film di Sorrentino. Ti va di andarci?
(it F) Volentieri! Ma prima passo dal medico.
(prompt M) Say: then rest. See you at nine in front of the cinema. || Allora riposati. Ci vediamo alle nove davanti al cinema.

(narrator) Now listen to the first conversation again.
(pause 1)
(it F) Che tempo da lupi! Piove da tre giorni.
(it M) Eh sì. E le previsioni dicono che domani nevicherà in montagna.
(it F) Non vedo l'ora di andare in vacanza. Se potessi, partirei domani!
(it M) Dove andrai quest'estate?
(it F) In Sardegna. Ma con questo clima pazzo, ho paura che faccia troppo caldo.
(it M) È vero, le estati sono sempre più calde. Il cambiamento climatico è un problema serio.
(it F) Cambiando argomento, hai provato il nuovo programma di intelligenza artificiale?
(it M) Sì, lo uso per scrivere il codice. È utilissimo, ma a volte sbaglia.
(it F) Secondo me, non sostituirà mai gli ingegneri.
(it M) Speriamo! Senti, stasera esce il nuovo film di Sorrentino. Ti va di andarci?
(it F) Volentieri! Ma prima passo dal medico. Ho un mal di gola terribile.
(it M) Allora riposati. Ci vediamo alle nove davanti al cinema.
(pause 1.5)
(narrator) This is the end of lesson seven. Listen to it again tomorrow. {A presto, e buona visione!}
"""

## 4 · Dry run (no audio) — check length before synthesizing

In [ ]:
tts = PimsleurTTS(CFG)
events = tts.parse(LESSON)
sec, counts = tts.estimate(events)
print(f"Estimated length: {sec/60:.1f} min   (PAUSE_SCALE={PAUSE_SCALE})")
print(counts)
print(f"Unique clips to synthesize: {len(tts.jobs_for(events))}")

## 5 · Build the MP3 + transcript

In [ ]:
mp3, txt = tts.build(LESSON, f"{LESSON_NAME}.mp3")
from IPython.display import Audio, display
display(Audio(mp3))
try:
    from google.colab import files
    files.download(mp3); files.download(txt)
except ImportError:
    pass

## 6 · Making the next lesson

Copy this notebook, change `LESSON_NAME`, and replace the `LESSON` script. A prompt that works well with an LLM:

> Write a ~45-minute Pimsleur-style Italian lesson at level **B1** on *[topic]*, in this tag format: `(narrator)`, `(it F)`, `(it M)`, `(repeat x2 slow)`, `(prompt) English || Italiano`, `(build) a | b | c`, `(pause n)`, `(section)`. Narration in plain short English sentences; put Italian words inside narration in `{braces}`. Start with a 10–14-line dialogue, break it down with back-chaining for long words, ~120 prompts, re-prompt each new sentence 2–4 times at growing intervals (spaced recall), recycle 10–15 sentences from Lesson *N–1*, include one role-play, and replay the dialogue at the end.

Tips
* Too fast/slow for you? Move `PAUSE_SCALE` (it scales every learner pause) — no need to edit the script.
* Long sentences cap at `MAX_ANSWER_MS`; raise it if you need more time on long prompts.
* `edge-tts` is an unofficial client of the Microsoft Edge read-aloud service; if it breaks, set `ENGINE="gtts"` (single Italian voice, no male/female distinction).
